# Operator search on MNIST: a tensor DAG writes its own classification equation

*ai · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/ai/operator_search_mnist)

Can a few readable tensor operators, wired as a DAG, compete with a CNN when there are only a handful of training images? Every edge of a small DAG holds a softmax-weighted mixture of 23 candidate operators (Sobel, Laplacian, Gabor, a learned 3×3 filter, ReLU, $x^2$, pooling, row sums, FFT magnitude, class templates, PCA, moments, linear and prototype read-outs); a DARTS search on 1 to 100 images per digit picks one operator per edge, and the DAG is printed as a LaTeX equation that reproduces its predictions exactly. Then data and compute against a LeNet-style CNN and two closed-form baselines, every seed shown.

**Vector form**

$$\bar o_e(z) = \sum_{o} \operatorname{softmax}(\alpha_e/T)_o\, o(z),\qquad z_i = \sum_{j<i} \bar o_{(j,i)}(z_j),\qquad \ell = \sum_{r,h} a_r b_h\, \mathrm{head}_h\!\Big(\frac{\mathrm{reduce}_r(\mathrm{pool}(z)) - m_r}{s_r}\Big),\qquad \hat y = \arg\max_k \ell_k$$

**Index (tensor) form**

$$w_{eo} = \frac{M_{eo}\, e^{\alpha_{eo}/T}}{\sum_{o'} M_{eo'}\, e^{\alpha_{eo'}/T}} \qquad z_{i,bhw} = \sum_{e \to i} \sum_o w_{eo}\, S_{o,bhw}(z_{j(e)}) \qquad \ell_{bk} = \sum_j W_{kj}\, \hat f_{bj} + b_k \qquad \theta \leftarrow \theta - \eta\, \partial_\theta \mathcal L_{\mathrm{train}},\ \ \alpha \leftarrow \alpha - \eta_\alpha\, \partial_\alpha \mathcal L_{\mathrm{val}}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $M_{eo}$ | `family_mask()` | (9, 23) | the typing rule: operator $o$ may sit on edge $e$ only if it maps that edge's input shape to its output shape (cell map → map, pool, reduce map → vector, head vector → logits) |
| $w_{eo}$ | `mixture_weights(alpha / T, mask)` | (9, 23) | masked softmax of the architecture tensor $\alpha$ (edges × operators); the temperature $T$ falls from 1 to 0.001 so the mixture ends one-hot |
| $S_{o}(z_j)$ | `conv2d(z, cat([bank, K]))` | (B, 10 + E_j, 14, 14) | every operator a node feeds, computed once for all its outgoing edges: one convolution gives Sobel-$x$, Sobel-$y$, Laplacian, Gabor and the learned $3\times 3$ filters $K_e$ |
| $z_i$ | `einsum('bol,eo->bel', S, w)` | (B, 1, 14, 14) | repeated $o$ ⇒ summed: each node is the sum of its mixed incoming edges; the cell output is the stack $(x, z_1, z_2, z_3)$ |
| $\hat f_r$ | `(reduce_op(r, pool(z)) - m) / s` | (B, D_r) | features of reduction $r$ (vec, row/column sums, $/\mathcal F/$, templates $\langle z, \mu_k\rangle$, PCA $U^\top \mathrm{vec}$, moments), standardised with the training images |
| $\ell_{bk}$ | `logits(Z, w, th, st)` | (B, 10) | all reductions × heads mixed at the logit level (their feature lengths differ); linear $W\hat f + b$ or prototypes $-e^\tau\lVert \hat f - p_k\rVert^2/D$, both started at the nearest class mean |
| $\partial_\alpha \mathcal L_{\mathrm{val}}$ | `autograd.grad(loss_val, [alpha])` | (9, 23) | first-order DARTS: $\theta$ steps on a training mini-batch, then $\alpha$ steps on a validation mini-batch |
| $\hat y$ | `evaluate_equation(write_equation(arch), x, th, st)` | (B,) | the discrete DAG written as LaTeX, read back by a small parser and evaluated: the same predictions as the DAG, exactly |

The question. This toy problem is Jacky Ka Long Ko's: an attempt at small-batch AI training with white-box equation writing. CNN versus a tensor method on MNIST; list the candidate tensor operators; let a tensor that encodes the DAG find the best operators and their wiring; train on small batches and compare the data and compute each needs for the same accuracy; and write the classification equation of the DAG found. Method. Images are 14×14 (MNIST averaged over 2×2 blocks, so 30 runs fit in about a minute of CPU). The supernet is a DARTS cell (Liu, Simonyan and Yang 2019) with three nodes $z_1, z_2, z_3$ fed by every earlier node through 6 edges, each a mixture of 12 cell operators (zero, identity, $x^2$, $|x|$, ReLU, tanh, soft threshold $S_\lambda$, learned $3\times 3$ filter, Sobel-$x$, Sobel-$y$, Laplacian, Gabor), then a pooling edge (average or max $2\times 2$), a reduction edge (vec, row sums, column sums, low-frequency FFT magnitude, class-template correlations, PCA projection, image moments) and a head edge (linear read-out or 10 prototypes). The architecture is the tensor $\alpha$ (9 edges × 23 operators) with a mask for the typing rule. Search: first-order DARTS, alternating a $\theta$ step on a mini-batch of the train half and an $\alpha$ step on the validation half, with the softmax temperature annealed from 1 to 0.001; the DAG keeps the argmax operator of every edge (zero removes the edge), inherits the trained $\theta$ and is fine-tuned briefly, the best validation step kept. Departures from DARTS, stated because they matter: the annealing (as in SNAS and GDAS) and the argmax rule close a discretisation gap that in a first trial cost up to 25 accuracy points; the image is kept as a fourth channel of the cell output; heads start at the nearest class mean in their feature space. The CNN (8 and 16 channels of $3\times 3$, max pooling, a linear layer: 2698 weights) is trained with Adam on the same images with the same early stopping. The baselines have no learned parameter: nearest class mean in pixels, and PCA followed by shrunk LDA. Results. The tests check that every operator equals its formula on a small input, that a one-hot mixture is the single operator and the one-hot supernet the discrete DAG, that discretisation picks the argmax, that the written equation, read back from its LaTeX alone, gives exactly the DAG's predictions, and that the baselines are their closed forms. Bundled run (CPU, one thread, 3 seeds), mean test accuracy for $n$ = 1, 5, 10, 50, 100 images per digit: CNN 0.35, 0.53, 0.64, 0.81, 0.85; searched DAG 0.38, 0.48, 0.52, 0.69, 0.82; PCA + LDA 0.41, 0.58, 0.66, 0.75, 0.76; nearest class mean 0.41, 0.56, 0.64, 0.68, 0.69. Images per digit to reach 70 %: CNN 18, DAG 52, PCA + LDA 20; to reach 80 %: CNN 45, DAG 91. Training time per run: CNN 0.4 to 1 s, search plus fine-tune 2 to 4 s. So, honestly: the CNN needs less data and less compute for the same accuracy from about 10 images per digit on; below that the closed-form baselines are best and no learner beats them; the structured model did not win at tiny $n$ (at $n$ = 1 it ties with the class means). Why: the search runs on a validation split that is itself tiny, so its choice is noisy, and its validation loss often prefers row sums (28 features, 290 learned numbers, a calmer loss) that classify worse than all pixels (about 2000 learned numbers); 7 of the 15 DAGs chose row sums. As reported for DARTS, the search favours parameter-free operators: Sobel filters fill most edges and a learned filter appears in 3 of 15 DAGs. Outside the bundled run, shorter searches (20 or 40 steps) chose all pixels and reached 0.74 to 0.79 at $n$ = 50, a hint that the longer search over-commits; this was seen on the test set, so it is an observation, not a tuned setting. The GPU (a GTX 1080 Ti) gave no speed-up: thousands of tiny kernels, not arithmetic, set the time. results/stages.csv has every number, results/equation.tex every equation found. Related work. Neural architecture search (Zoph and Le 2017) and its differentiable form DARTS; symbolic regression, which searches formulas instead of networks (AI Feynman, Udrescu and Tegmark 2020; PySR, Cranmer 2023; genetic programming, Koza 1992); LeNet (LeCun et al. 1998) for the CNN. Data: MNIST (LeCun, Cortes and Burges), licence CC BY-SA 3.0; data/mnist_subset.npz holds 1000 training-pool and 2000 test images (the share-alike licence applies to this subset), see data/SOURCES.md. Try it. Change show_n to see the DAG found with 1 or 100 images per digit, more search steps, or one seed for a quick run.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/ai/operator_search_mnist/main.py
"""Main loop: for every training-set size n and seed, search a tensor DAG (DARTS supernet → discrete DAG → equation)
and train a CNN on the same n images per class; compare accuracy, data and time; write the tables and the animation."""

import csv
import math
import time
from pathlib import Path

import numpy as np
import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve
from . import tensors as T

TARGETS = (0.7, 0.8)  # accuracies for "images per class needed"


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    fast = fast_mode()
    torch.manual_seed(0)
    data = T.load(dev)
    sizes = (1, 10) if fast else T.SIZES
    seeds = 1 if fast else params["seeds"]
    cfg = {"search_steps": 15 if fast else params["search_steps"],
           "retrain_steps": 10 if fast else params["retrain_steps"],
           "cnn_steps": 20 if fast else params["cnn_steps"], "batch": params["batch"]}  # fmt: skip
    show = params["show_n"] if params["show_n"] in sizes else sizes[-1]
    print(f"MNIST subset: pool {T.N_POOL} images (100 per digit), test {len(data['y_test'])} (200 per digit), "
          f"14×14 pixels; device {dev}; sizes n = {list(sizes)} per class, {seeds} seed(s)")  # fmt: skip
    print(f"CNN: {T.cnn_param_count()} weights; supernet: 6 cell edges × 12 operators, pool 2, reduce 7, head 2; "
          f"search {cfg['search_steps']} steps, fine-tune {cfg['retrain_steps']}, CNN ≤ {cfg['cnn_steps']} steps, "
          f"mini-batch ≤ {cfg['batch']}")  # fmt: skip

    tr, va = T.small_set(data["y_pool"], 1, 0)  # warm-up (first-call overheads), so no method pays them in its timing
    warm = {"search_steps": 2, "retrain_steps": 2, "cnn_steps": 2, "batch": cfg["batch"]}
    solve.run_cnn(data, tr, va, warm, 0), solve.run_tensor(data, tr, va, warm, 0)
    rows, showcase, t_all = [], None, time.perf_counter()
    for n in sizes:
        for seed in range(seeds):
            tr, va = T.small_set(data["y_pool"], n, seed)
            c = solve.run_cnn(data, tr, va, cfg, seed)
            d = solve.run_tensor(data, tr, va, cfg, seed)
            b = solve.baselines(data, tr, va)
            a = d["arch"]
            arch_txt = " ".join(f"{e + 1}:{op}" for e, op in a["edges"]) + f" | {a['pool']} {a['reduce']} {a['head']}"
            rows += [
                {"n": n, "seed": seed, "method": "cnn", "acc": c["acc"], "seconds": c["seconds"], "params": c["params"],
                 "constants": 0, "arch": "conv8-pool-conv16-pool-linear"},
                {"n": n, "seed": seed, "method": "tensor_dag", "acc": d["acc"], "seconds": d["seconds"],
                 "params": d["params"], "constants": d["constants"], "arch": arch_txt, "equation": d["equation"]},
                {"n": n, "seed": seed, "method": "supernet", "acc": d["acc_supernet"], "seconds": "", "params": "",
                 "constants": "", "arch": "mixture before discretisation"},
                {"n": n, "seed": seed, "method": "nearest_class_mean", "acc": b["ncm"], "seconds": "", "params": 0,
                 "constants": 10 * 196, "arch": "closed form"},
                {"n": n, "seed": seed, "method": "pca_lda", "acc": b["pca_lda"], "seconds": "", "params": 0,
                 "constants": "", "arch": "closed form"},
            ]  # fmt: skip
            print(f"n = {n:3d} seed {seed}: CNN {c['acc']:.3f} ({c['seconds']:.1f} s) | tensor DAG {d['acc']:.3f} "
                  f"(supernet {d['acc_supernet']:.3f}, {d['seconds']:.1f} s, {d['params']} learned) | "
                  f"class means {b['ncm']:.3f}, PCA+LDA {b['pca_lda']:.3f} | {arch_txt}", flush=True)  # fmt: skip
            if n == show and seed == 0:
                showcase = {"n": n, **d, "cnn_pred": c["pred"]}

    # data needed and time at equal accuracy
    mean = {m: [np.mean([r["acc"] for r in rows if r["n"] == n and r["method"] == m]) for n in sizes]
            for m in ("cnn", "tensor_dag", "nearest_class_mean", "pca_lda")}  # fmt: skip
    secs = {m: [np.mean([r["seconds"] for r in rows if r["n"] == n and r["method"] == m]) for n in sizes]
            for m in ("cnn", "tensor_dag")}  # fmt: skip
    need = {(m, t): solve.n_to_reach(sizes, mean[m], t) for m in mean for t in TARGETS}
    for t in TARGETS:
        print(f"images per class to reach {t:.0%} test accuracy (mean over seeds, log-linear between sizes): " +
              ", ".join(f"{m} {'never' if math.isnan(need[m, t]) else f'{need[m, t]:.0f}'}" for m in mean))  # fmt: skip
    for k, n in enumerate(sizes):
        print(f"n = {n:3d}: mean accuracy CNN {mean['cnn'][k]:.3f} vs tensor DAG {mean['tensor_dag'][k]:.3f}; "
              f"mean time {secs['cnn'][k]:.1f} s vs {secs['tensor_dag'][k]:.1f} s")  # fmt: skip

    res = out_dir / "results"
    res.mkdir(parents=True, exist_ok=True)
    with open(res / "stages.csv", "w", newline="", encoding="utf-8") as f:
        f.write(f"# operator_search_mnist: one row per (n images per class, seed, method); test set 2000 images; "
                f"device {dev}; seconds = wall time of training (search + fine-tune for the tensor DAG)\n")  # fmt: skip
        w = csv.writer(f)
        w.writerow(["n_per_class", "seed", "method", "test_accuracy", "train_seconds", "learned_parameters",
                    "data_constants", "architecture"])  # fmt: skip
        for r in rows:
            w.writerow([r["n"], r["seed"], r["method"], f"{r['acc']:.4f}",
                        r["seconds"] if r["seconds"] == "" else f"{r['seconds']:.3f}", r["params"], r["constants"],
                        r["arch"]])  # fmt: skip
        f.write("# images per class needed to reach a mean test accuracy (log-linear interpolation; nan = never)\n")
        for (m, t), v in need.items():
            w.writerow(["needed", f"target {t:.2f}", m, "", "", "", "", f"{v:.2f}"])
    with open(res / "equation.tex", "w", encoding="utf-8") as f:
        f.write("% Classification equations found by the DARTS search on MNIST (14x14), one per (n, seed).\n")
        f.write(
            "% S_x, S_y Sobel, \\Delta Laplacian, G Gabor, K_e learned 3x3 filter of edge e, * cross-correlation,\n"
        )
        f.write(
            "% S_\\lambda soft threshold, z = (x, z_1, z_2, z_3) stacked; m, s, mu_k, U from the training images.\n"
        )
        for r in [r for r in rows if r["method"] == "tensor_dag"]:
            f.write(f"% n = {r['n']} per class, seed {r['seed']}: test accuracy {r['acc']:.3f}\n")
            f.write("\\begin{aligned}\n" + " \\\\\n".join(r["equation"]) + "\n\\end{aligned}\n\n")
    print(f"the equation of the DAG found with n = {show} (seed 0):")
    for line in showcase["equation"]:
        print("   ", line)
    print(f"study {time.perf_counter() - t_all:.1f} s on {dev}; animation next{' (fast mode)' if fast else ''}")
    return plot.animate(rows, sizes, showcase, data, need, TARGETS, out_dir / "operator_search_mnist.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/ai/operator_search_mnist/equation.py
r"""The operator library as tensor operations, each with the formula the equation writer prints, and the typed
supernet: a DAG whose every edge is a softmax(α)-weighted mixture of all the operators of its family.

Typing rule (what makes every mixture well defined: all operators of one family map the same input shape to the same
output shape, so their outputs can be added):

    cell family     map → map      (B, 1, 14, 14) → (B, 1, 14, 14)   0, identity, x², |x|, ReLU, tanh, S_λ, K∗, Sobel-x,
                                                                       Sobel-y, Laplacian, Gabor
    pool family     maps → maps    (B, 4, 14, 14) → (B, 4, 7, 7)     avgpool₂, maxpool₂
    reduce family   maps → vector  (B, 4, 7, 7) → (B, D_r)           vec, rowsum, colsum, |F|₄, ⟨·, μ_k⟩, Uᵀvec, M_pq
    head family     vector → logits (B, D_r) → (B, 10)               linear W f̂ + b, prototypes −e^τ‖f̂ − p_k‖²/D

A reduction's output length D_r differs between reductions, so each reduction carries its own heads and the mixture
is taken over the logits: ℓ = Σ_r a_r Σ_h b_h head_{r,h}(f̂_r). The cell is a DARTS cell: nodes z_1, z_2, z_3, each
the sum of the mixed edges from every earlier node (x = z_0 is the image); its output is the stack z = (x, z_1, z_2,
z_3): DARTS stacks only z_1 … z_3; the image itself is kept as a fourth channel (a skip connection), so the
pixels always reach the read-out.

    mixed edge     ō_e(z) = Σ_o softmax(α_e)_o o(z)          (masked softmax: α_eo = −∞ outside the edge's family)
    node           z_i = Σ_{j<i} ō_{(j,i)}(z_j)
    standardise    f̂ = (f − m) / s                            (m, s: feature mean and std over the training images)
    loss           L = −(1/B) Σ_b log softmax(ℓ_b)_{y_b}
"""

import functools
import math

import torch
import torch.nn.functional as F

from . import tensors as T

CELL_OPS = ("zero", "identity", "square", "abs", "relu", "tanh", "shrink", "conv3", "sobel_x", "sobel_y", "laplace",
            "gabor")  # fmt: skip
POOL_OPS = ("avgpool", "maxpool")
RED_OPS = ("vec", "rowsum", "colsum", "fft", "template", "pca", "moments")
HEAD_OPS = ("linear", "proto")
OPS = CELL_OPS + POOL_OPS + RED_OPS + HEAD_OPS  # the 23 columns of α
FIXED = ("sobel_x", "sobel_y", "laplace", "gabor")
N_NODES = 3
CHANNELS = N_NODES + 1  # the cell output (x, z_1, z_2, z_3)
CELL_EDGES = tuple((j, i) for i in range(1, N_NODES + 1) for j in range(i))  # (0,1) (0,2) (1,2) (0,3) (1,3) (2,3)
EDGES = tuple(f"z{j}→z{i}" for j, i in CELL_EDGES) + ("pool", "reduce", "head")  # the 9 rows of α
SHORT = {"zero": "0", "identity": "id", "square": "x²", "abs": "|x|", "relu": "ReLU", "tanh": "tanh",
         "shrink": "soft thresh.",
         "conv3": "K∗ (3×3)", "sobel_x": "Sobel-x", "sobel_y": "Sobel-y", "laplace": "Laplacian", "gabor": "Gabor",
         "avgpool": "avg-pool 2", "maxpool": "max-pool 2", "vec": "vec", "rowsum": "row sums", "colsum": "col sums",
         "fft": "|FFT|", "template": "⟨z, μ_k⟩", "pca": "PCA", "moments": "moments", "linear": "linear",
         "proto": "prototypes"}  # fmt: skip

# The formula of every operator, written into the equation: {a} is the argument, {e} the edge number (filters K_e).
LATEX = {
    "zero": "0",
    "identity": "{a}",
    "square": "({a})^{{2}}",
    "abs": "|{a}|",
    "relu": r"\mathrm{{ReLU}}({a})",
    "tanh": r"\tanh({a})",
    "shrink": r"S_{{\lambda}}({a})",
    "conv3": r"K_{{{e}}} \ast {a}",
    "sobel_x": r"S_{{x}} \ast {a}",
    "sobel_y": r"S_{{y}} \ast {a}",
    "laplace": r"\Delta \ast {a}",
    "gabor": r"G \ast {a}",
    "avgpool": r"\mathrm{{avgpool}}_{{2}}({a})",
    "maxpool": r"\mathrm{{maxpool}}_{{2}}({a})",
    "vec": r"\mathrm{{vec}}({a})",
    "rowsum": r"\mathrm{{rowsum}}({a})",
    "colsum": r"\mathrm{{colsum}}({a})",
    "fft": r"|\mathcal{{F}}({a})|_{{4}}",
    "template": r"\langle {a}, \mu_{{k}} \rangle",
    "pca": r"U^{{\top}} \mathrm{{vec}}({a})",
    "moments": r"M_{{pq}}({a})",
}


def family_mask() -> torch.Tensor:
    """M (9, 23) bool: M_eo = operator o may sit on edge e (the typing rule)."""
    M = torch.zeros(len(EDGES), len(OPS), dtype=torch.bool)
    for e, name in enumerate(EDGES):
        fam = {"pool": POOL_OPS, "reduce": RED_OPS, "head": HEAD_OPS}.get(name, CELL_OPS)
        M[e, [OPS.index(o) for o in fam]] = True
    return M


def mixture_weights(alpha: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """softmax over each edge's own family: w_eo = exp α_eo / Σ_{o' ∈ family(e)} exp α_eo', 0 outside it. (9, 23)."""
    return torch.softmax(alpha.masked_fill(~mask, float("-inf")), dim=1)


# ---------------------------------------------------------------- cell operators: map (B, 1, H, W) → map
def cell_op(name: str, z: torch.Tensor, K: torch.Tensor | None = None) -> torch.Tensor:
    """One cell operator applied to z (B, 1, H, W); K (3, 3) is the edge's learned filter (conv3 only).
    ∗ is the 2D cross-correlation with zero padding (what CNN layers compute)."""
    if name == "zero":
        return torch.zeros_like(z)
    if name == "identity":
        return z
    if name == "square":
        return z * z
    if name == "abs":
        return z.abs()
    if name == "relu":
        return z.clamp(min=0)
    if name == "tanh":
        return torch.tanh(z)
    if name == "shrink":  # soft threshold S_λ(z) = sign(z) max(|z| − λ, 0)
        return torch.sign(z) * (z.abs() - T.LAMBDA).clamp(min=0)
    if name == "conv3":
        return F.conv2d(z, K[None, None], padding=1)
    k = _bank(z.device, z.dtype)[FIXED.index(name)]
    return F.conv2d(z, k[None, None], padding=2)


@functools.lru_cache(maxsize=8)
def _bank(dev: torch.device, dtype: torch.dtype) -> torch.Tensor:
    return T.fixed_kernels().to(dev, dtype)


@functools.lru_cache(maxsize=8)
def _moments(h: int, dev: torch.device, dtype: torch.dtype) -> torch.Tensor:
    return T.moment_basis(h).to(dev, dtype)


def _pattern(w: torch.Tensor) -> list[list[str]]:
    """Per entry of w: "0" (skip), "1" (use as is) or "w" (multiply); read once on the CPU (no per-op GPU sync)."""
    wc = w.detach().cpu()
    return [["0" if v == 0 else "1" if v == 1 else "w" for v in row] for row in wc.tolist()]


def mixed_edge(z: torch.Tensor, w: torch.Tensor, K: torch.Tensor, pat: list[str] | None = None) -> torch.Tensor:
    """ō(z) = Σ_o w_o o(z) over the cell family; w (23,) one row of the mixture weights. Operators with w_o = 0 are not
    evaluated (a one-hot row is then exactly that one operator)."""
    pat = pat if pat is not None else _pattern(w[None])[0]
    out = None
    for i, name in enumerate(CELL_OPS):
        if pat[i] != "0" and name != "zero":
            term = cell_op(name, z, K) if pat[i] == "1" else w[i] * cell_op(name, z, K)
            out = term if out is None else out + term
    return torch.zeros_like(z) if out is None else out


ELEMENTWISE = ("identity", "square", "abs", "relu", "tanh", "shrink")


def cell(x: torch.Tensor, w: torch.Tensor, K: torch.Tensor) -> torch.Tensor:
    """z_0 = x; z_i = Σ_{j<i} ō_(j,i)(z_j); returns the stack (x, z_1, z_2, z_3), (B, 4, H, W). w (9, 23), K (6, 3, 3).

    A discrete DAG (every weight 0 or 1) is evaluated edge by edge with cell_op, exactly as the written equation is.
    A mixture is batched per source node j: ONE convolution with the stacked kernels gives all four fixed filters and
    the learned 3×3 filters (zero-padded to 5×5) of all edges leaving j; with the six element-wise operators that is
    the stack S (B, 10, HW) of parameter-free outputs, and ō_e = Σ_o w_eo S_o + w_e,conv K_e ∗ z_j for all those
    edges in one einsum."""
    pat = _pattern(w)
    if all(p in ("0", "1") for row in pat[: len(CELL_EDGES)] for p in row):
        z = [x]
        for i in range(1, N_NODES + 1):
            node = None
            for e, (j, ii) in enumerate(CELL_EDGES):
                op = next((CELL_OPS[o] for o in range(1, len(CELL_OPS)) if pat[e][o] == "1"), None)
                if ii == i and op is not None:
                    t = cell_op(op, z[j], K[e])
                    node = t if node is None else node + t
            z.append(torch.zeros_like(x) if node is None else node)
        return torch.cat(z, dim=1)

    B, _, H, W_ = x.shape
    bank = _bank(x.device, x.dtype)  # (4, 5, 5)
    cols = [OPS.index(n) for n in ELEMENTWISE + FIXED]
    ic = OPS.index("conv3")
    z, contrib = [x], {}
    for j in range(N_NODES):
        out_e = [e for e, (jj, _) in enumerate(CELL_EDGES) if jj == j]
        Kp = F.pad(K[out_e], (1, 1, 1, 1))  # (E_j, 5, 5): 3×3 filters centred in 5×5
        R = F.conv2d(z[j], torch.cat([bank, Kp])[:, None], padding=2).flatten(2)  # (B, 4 + E_j, HW)
        zf = z[j].flatten(1)[:, None]  # (B, 1, HW)
        S = torch.cat([cell_op(n, zf) for n in ELEMENTWISE] + [R[:, :4]], 1)  # (B, 10, HW)
        mix = torch.einsum("bol,eo->bel", S, w[out_e][:, cols]) + R[:, 4:] * w[out_e, ic][None, :, None]
        for k, e in enumerate(out_e):
            contrib[e] = mix[:, k].view(B, 1, H, W_)
        z.append(sum(contrib[e] for e, (_, i) in enumerate(CELL_EDGES) if i == j + 1))
    return torch.cat(z, dim=1)


# ---------------------------------------------------------------- pool family: (B, C, H, W) → (B, C, H/2, W/2)
def pool_op(name: str, z: torch.Tensor) -> torch.Tensor:
    return F.avg_pool2d(z, 2) if name == "avgpool" else F.max_pool2d(z, 2)


# ---------------------------------------------------------------- reduce family: (B, C, h, h) → (B, D_r)
def reduce_op(name: str, Z: torch.Tensor, stats: dict | None = None) -> torch.Tensor:
    """vec: all pixels; rowsum_ch = Σ_w Z_chw; colsum_cw = Σ_h Z_chw; |F(Z)|_cuv = |Σ_hw Z_chw e^{−2πi(uh+vw)/h}|/h for
    u, v < 4; template ⟨Z_c, μ_kc⟩ = Σ_hw Z_chw μ_kchw (μ_k the class means of Z over the training images);
    PCA (Uᵀ vec Z)_cq = Σ_p U_cpq Z_cp (U_c the top principal axes of channel c over the training images);
    moments M_pq(Z)_c = mean_hw u^p v^q Z_chw for (p, q) ∈ {00, 10, 01, 20, 02, 11}, u, v ∈ [−1, 1]."""
    if name == "vec":
        return Z.flatten(1)
    if name == "rowsum":
        return Z.sum(3).flatten(1)
    if name == "colsum":
        return Z.sum(2).flatten(1)
    if name == "fft":
        return torch.fft.fft2(Z, norm="ortho").abs()[..., : T.FFT_KEEP, : T.FFT_KEEP].flatten(1)
    if name == "template":
        return torch.einsum("bchw,kchw->bck", Z, stats["mu"]).flatten(1)
    if name == "pca":
        return torch.einsum("bcp,cpq->bcq", Z.flatten(2), stats["U"]).flatten(1)
    if name == "moments":
        return torch.einsum("bchw,phw->bcp", Z, _moments(Z.shape[-1], Z.device, Z.dtype)).flatten(1) / Z[0, 0].numel()
    raise ValueError(name)


def reduce_dims(channels: int = CHANNELS, h: int = 7) -> dict:
    """D_r, the feature length of each reduction for C channels of h × h."""
    return {"vec": channels * h * h, "rowsum": channels * h, "colsum": channels * h,
            "fft": channels * T.FFT_KEEP**2, "template": channels * T.CLASSES, "pca": channels * T.Q_PCA,
            "moments": channels * 6}  # fmt: skip


def class_means(X: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """μ_k = mean of the rows of X with label k, as one product with the one-hot labels: (N, ...) → (10, ...)."""
    Y = F.one_hot(y, T.CLASSES).to(X.dtype)  # (N, 10)
    mu = Y.T @ X.flatten(1) / Y.sum(0)[:, None].clamp(min=1)
    return mu.view(T.CLASSES, *X.shape[1:])


def data_statistics(Z: torch.Tensor, y: torch.Tensor, reds: list[str]) -> dict:
    """The data-derived constants (no gradient) from the pooled maps Z (N, C, h, h) of the training images: class
    means μ (10, C, h, h), principal axes U (C, h², q) with a fixed sign (largest entry positive), the feature mean m_r
    (D_r,) and ONE scale per channel (the RMS deviation over that channel's features, repeated over its block: a
    per-feature std from a few images would blow up pixels that happen to be blank), and the class means
    c_r (10, D_r) of f̂_r = (f_r − m_r)/s_r."""
    Z = Z.detach()
    st = {"mu": class_means(Z, y)}
    Zc = Z.flatten(2) - Z.flatten(2).mean(0)  # (N, C, P)
    _, _, Vh = torch.linalg.svd(Zc.transpose(0, 1), full_matrices=False)  # (C, min(N,P), P)
    U = Vh[:, : T.Q_PCA].transpose(1, 2)  # (C, P, q')
    if U.shape[2] < T.Q_PCA:  # fewer images than components: pad with zero axes
        U = torch.cat([U, U.new_zeros(U.shape[0], U.shape[1], T.Q_PCA - U.shape[2])], 2)
    sign = torch.sign(torch.gather(U, 1, U.abs().argmax(1, keepdim=True)))
    st["U"] = U * torch.where(sign == 0, torch.ones_like(sign), sign)
    for r in reds:
        f = reduce_op(r, Z, st)
        m = f.mean(0)
        C = Z.shape[1]
        s = ((f - m) ** 2).view(len(f), C, -1).mean((0, 2)).sqrt().clamp(min=1e-6)  # (C,) one scale per channel
        s = s.repeat_interleave(f.shape[1] // C)  # (D,) constant on each channel's block of features
        st[f"m_{r}"], st[f"s_{r}"] = m, s
        st[f"c_{r}"] = class_means((f - m) / s, y)
    return st


def init_heads(th: dict, st: dict, reds: list[str], scale: float = 3.0) -> None:
    """Start every head at the nearest class mean in its feature space (the parameter-free baseline): prototypes
    p_k = c_k, e^τ = scale; linear W_k = 2 scale c_k / D, b_k = −scale ‖c_k‖² / D (the same decisions and logits up to
    a k-independent term). Training then learns a correction to it."""
    with torch.no_grad():
        for r in reds:
            c = st[f"c_{r}"]
            D = c.shape[1]
            th[f"P_{r}"].copy_(c)
            th[f"tau_{r}"].fill_(math.log(scale))
            th[f"W_{r}"].copy_(2 * scale * c / D)
            th[f"b_{r}"].copy_(-scale * (c**2).sum(1) / D)


# ---------------------------------------------------------------- head family: f̂ (B, D) → logits (B, 10)
def head_op(name: str, fh: torch.Tensor, th: dict, r: str) -> torch.Tensor:
    """linear: ℓ_k = Σ_j W_kj f̂_j + b_k;  proto: ℓ_k = −e^τ ‖f̂ − p_k‖² / D (argmax ℓ = nearest prototype)."""
    if name == "linear":
        return torch.einsum("bj,kj->bk", fh, th[f"W_{r}"]) + th[f"b_{r}"]
    d2 = ((fh[:, None, :] - th[f"P_{r}"][None]) ** 2).sum(-1)
    return -torch.exp(th[f"tau_{r}"]) * d2 / fh.shape[1]


def pooled(x: torch.Tensor, w: torch.Tensor, th: dict) -> torch.Tensor:
    """The mixed cell and the mixed pool: (B, 1, 14, 14) → (B, 4, 7, 7)."""
    z = cell(x, w, th["K"])
    e = EDGES.index("pool")
    pat = _pattern(w[e : e + 1])[0]
    out = None
    for name in POOL_OPS:
        o = OPS.index(name)
        if pat[o] != "0":
            t = pool_op(name, z) if pat[o] == "1" else w[e, o] * pool_op(name, z)
            out = t if out is None else out + t
    return out


def logits(Z: torch.Tensor, w: torch.Tensor, th: dict, st: dict) -> torch.Tensor:
    """ℓ = Σ_r a_r Σ_h b_h head_{r,h}(f̂_r), f̂_r = (reduce_r(Z) − m_r)/s_r; reductions with a_r = 0 are skipped.
    The mixture is batched over the reductions: the f̂_r are concatenated (B, ΣD_r), each block scaled by its a_r,
    so all linear heads are one product and all prototype heads one weighted sum of squared differences."""
    er, eh = EDGES.index("reduce"), EDGES.index("head")
    pat = _pattern(w)
    reds = [r for r in RED_OPS if pat[er][OPS.index(r)] != "0"]
    heads = [h for h in HEAD_OPS if pat[eh][OPS.index(h)] != "0"]
    fh = [(reduce_op(r, Z, st) - st[f"m_{r}"]) / st[f"s_{r}"] for r in reds]
    if len(reds) == len(heads) == 1 and pat[er][OPS.index(reds[0])] == pat[eh][OPS.index(heads[0])] == "1":
        return head_op(heads[0], fh[0], th, reds[0])  # one discrete DAG
    D = torch.tensor([f.shape[1] for f in fh], device=Z.device)
    a = w[er, [OPS.index(r) for r in reds]]  # (R,)
    f_all = torch.cat(fh, 1)  # (B, ΣD)
    out = 0.0
    if "linear" in heads:
        W_all = torch.cat([th[f"W_{r}"] for r in reds], 1)  # (10, ΣD)
        b_all = torch.stack([th[f"b_{r}"] for r in reds])  # (R, 10)
        lin = torch.einsum("bj,kj->bk", f_all * a.repeat_interleave(D), W_all) + a @ b_all
        out = out + w[eh, OPS.index("linear")] * lin
    if "proto" in heads:
        P_all = torch.cat([th[f"P_{r}"] for r in reds], 1)
        tau = torch.stack([th[f"tau_{r}"] for r in reds])
        coef = (a * torch.exp(tau) / D).repeat_interleave(D)  # (ΣD,) a_r e^{τ_r} / D_r on block r
        d2 = (f_all[:, None, :] - P_all[None]) ** 2  # (B, 10, ΣD)
        out = out - w[eh, OPS.index("proto")] * torch.einsum("bkj,j->bk", d2, coef)
    return out


def forward(x: torch.Tensor, w: torch.Tensor, th: dict, st: dict) -> torch.Tensor:
    """The whole supernet (or, with one-hot rows of w, one discrete DAG): images (B, 1, 14, 14) → logits (B, 10)."""
    return logits(pooled(x, w, th), w, th, st)


# ---------------------------------------------------------------- the CNN and the parameter-free baselines
def cnn(x: torch.Tensor, p: dict) -> torch.Tensor:
    """conv 3×3 → ReLU → max-pool 2 → conv 3×3 → ReLU → max-pool 2 → linear: (B, 1, 14, 14) → (B, 10)."""
    h = F.max_pool2d(F.relu(F.conv2d(x, p["W1"], p["b1"], padding=1)), 2)
    h = F.max_pool2d(F.relu(F.conv2d(h, p["W2"], p["b2"], padding=1)), 2)
    return h.flatten(1) @ p["W3"].T + p["b3"]


def nearest_class_mean(x_train: torch.Tensor, y_train: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    """ŷ = argmin_k ‖x − μ_k‖², μ_k the mean training image of digit k (no learned parameter)."""
    X, Q = x_train.flatten(1), x.flatten(1)
    mu = class_means(X, y_train)  # (10, 196)
    return torch.cdist(Q, mu).argmin(1)


def pca_lda(x_train: torch.Tensor, y_train: torch.Tensor, x: torch.Tensor, q: int = 40, shrink: float = 0.1):
    """Closed form: PCA to q ≤ N − 1 axes, then LDA with a shrunk within-class covariance
    Σ = (1 − γ) S_w + γ tr(S_w)/q I (γ = 0.1; with one image per class S_w = 0 → Σ = I, nearest class mean in PCA
    space). δ_k(x) = μ_kᵀ Σ⁻¹ x − ½ μ_kᵀ Σ⁻¹ μ_k, equal priors."""
    X = x_train.flatten(1).double()
    mean = X.mean(0)
    _, _, Vh = torch.linalg.svd(X - mean, full_matrices=False)
    P = Vh[: min(q, len(X) - 1)].T  # (784, q)
    Y, Qx = (X - mean) @ P, (x.flatten(1).double() - mean) @ P
    mu = class_means(Y, y_train)
    R = Y - mu[y_train]
    Sw = R.T @ R / len(Y)
    tr = float(torch.trace(Sw)) / len(Sw)
    Sig = (1 - shrink) * Sw + (shrink * tr if tr > 0 else 1.0) * torch.eye(len(Sw), dtype=Sw.dtype, device=Sw.device)
    A = torch.linalg.solve(Sig, mu.T)  # Σ⁻¹ μ_k as columns (q, 10)
    return (Qx @ A - 0.5 * (mu * A.T).sum(1)).argmax(1)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/ai/operator_search_mnist/tensors.py
"""Build the concrete tensors: the bundled MNIST subset, the balanced small training sets, the fixed filter bank and the
fresh (untrained) parameters of the supernet and of the CNN.

data/mnist_subset.npz holds `images` (3000, 28, 28) uint8 and `labels` (3000,) uint8: the first 100 images of every
digit in the MNIST training file (rows 0–999, a pool from which the small training sets are drawn) followed by the
first 200 of every digit in the MNIST test file (rows 1000–2999, the fixed test set). build_subset() made it once
from the raw MNIST files, read with gzip and numpy only (the raw folder is untrusted data, nothing in it is executed).
"""

import gzip
import math
from pathlib import Path

import numpy as np
import torch

DATA = Path(__file__).parent / "data" / "mnist_subset.npz"
POOL_PER_CLASS, TEST_PER_CLASS, CLASSES = 100, 200, 10
N_POOL = POOL_PER_CLASS * CLASSES  # rows [0, N_POOL) are the training pool, the rest the test set
SIZES = (1, 5, 10, 50, 100)  # training images per class in the data study
LAMBDA = 0.25  # soft-threshold level S_λ (pixels are in [0, 1])
Q_PCA = 8  # principal components kept per channel
FFT_KEEP = 4  # |F(Z)|_{uv} kept for 0 ≤ u, v < 4 (the low frequencies)
MD5 = {  # the published checksums of the raw files (verified before use)
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}


def _idx(path: Path) -> np.ndarray:
    """Read one IDX file (big-endian header: magic, dims; then uint8 data) with gzip + numpy only."""
    raw = gzip.decompress(path.read_bytes())
    ndim = raw[3]
    shape = tuple(int.from_bytes(raw[4 + 4 * i : 8 + 4 * i], "big") for i in range(ndim))
    return np.frombuffer(raw, dtype=np.uint8, offset=4 + 4 * ndim).reshape(shape)


def build_subset(raw_dir: Path, out: Path = DATA) -> Path:
    """One-off: the first POOL_PER_CLASS training and TEST_PER_CLASS test images of every digit, in file order."""
    import hashlib

    for name, md5 in MD5.items():
        assert hashlib.md5((raw_dir / name).read_bytes()).hexdigest() == md5, f"checksum mismatch: {name}"
    parts = []
    for prefix, per in (("train", POOL_PER_CLASS), ("t10k", TEST_PER_CLASS)):
        x, y = _idx(raw_dir / f"{prefix}-images-idx3-ubyte.gz"), _idx(raw_dir / f"{prefix}-labels-idx1-ubyte.gz")
        keep = np.sort(np.concatenate([np.flatnonzero(y == c)[:per] for c in range(CLASSES)]))
        parts.append((x[keep], y[keep]))
    np.savez_compressed(out, images=np.concatenate([p[0] for p in parts]), labels=np.concatenate([p[1] for p in parts]))
    return out


def load(device: torch.device) -> dict:
    """Pool and test set as float images in [0, 1] at half resolution, (N, 1, 14, 14) (each pixel the mean of a 2×2
    block: the whole 30-run study then fits in about a minute of one CPU core), and int64 labels (N,)."""
    with np.load(DATA) as d:
        x = torch.from_numpy(d["images"].astype(np.float32) / 255.0)[:, None]
        x = torch.nn.functional.avg_pool2d(x, 2).to(device)
        y = torch.from_numpy(d["labels"].astype(np.int64)).to(device)
    return {"x_pool": x[:N_POOL], "y_pool": y[:N_POOL], "x_test": x[N_POOL:], "y_test": y[N_POOL:]}


def small_set(y_pool: torch.Tensor, n: int, seed: int) -> tuple[torch.Tensor, torch.Tensor]:
    """n images per class drawn (without replacement) from the pool, split per class into a train half (⌈n/2⌉) and
    a validation half (⌊n/2⌋). With n = 1 there is no second image: train and validation are the same 10 images."""
    gen = torch.Generator().manual_seed(1000 * seed + n)
    tr, va = [], []
    for c in range(CLASSES):
        idx = torch.nonzero(y_pool.cpu() == c).flatten()
        pick = idx[torch.randperm(len(idx), generator=gen)[:n]]
        tr.append(pick[: math.ceil(n / 2)])
        va.append(pick[math.ceil(n / 2) :] if n > 1 else pick)
    return torch.cat(tr).to(y_pool.device), torch.cat(va).to(y_pool.device)


def balanced_batch(y: torch.Tensor, size: int, gen: torch.Generator) -> torch.Tensor:
    """Indices of a class-balanced mini-batch of about `size` images (all of them when the set is smaller). y must be
    class-blocked with equal counts, as small_set returns it: class c occupies rows c m … c m + m − 1."""
    m = len(y) // CLASSES
    if len(y) <= size:
        return torch.arange(len(y), device=y.device)
    per = max(1, size // CLASSES)
    pick = torch.rand(CLASSES, m, generator=gen).argsort(1)[:, :per] + m * torch.arange(CLASSES)[:, None]
    return pick.flatten().to(y.device)


def fixed_kernels() -> torch.Tensor:
    """The fixed filter bank (4, 5, 5): Sobel-x, Sobel-y, Laplacian (3×3, zero-padded to 5×5) and a zero-mean Gabor
    (θ = 45°, wavelength 4 px, σ = 1.5 px), applied as cross-correlations with zero padding, as in CNNs."""
    sx = torch.tensor([[-1.0, 0, 1], [-2, 0, 2], [-1, 0, 1]])
    lap = torch.tensor([[0.0, 1, 0], [1, -4, 1], [0, 1, 0]])
    u = torch.arange(-2.0, 3.0)
    U, V = torch.meshgrid(u, u, indexing="ij")
    g = torch.exp(-(U**2 + V**2) / (2 * 1.5**2)) * torch.cos(2 * math.pi * (U + V) / math.sqrt(2) / 4)
    g = g - g.mean()
    g = g / g.abs().sum() * 4  # same total weight as the Sobel kernel
    bank = torch.zeros(4, 5, 5)
    for k, m in enumerate((sx, sx.T.contiguous(), lap)):
        bank[k, 1:4, 1:4] = m
    bank[3] = g
    return bank


def moment_basis(h: int) -> torch.Tensor:
    """(6, h, h): 1, u, v, u², v², uv on the pixel grid with u, v ∈ [−1, 1] (u along rows, v along columns)."""
    u = torch.linspace(-1, 1, h)
    U, V = torch.meshgrid(u, u, indexing="ij")
    return torch.stack([torch.ones_like(U), U, V, U**2, V**2, U * V])


def init_supernet(n_edges: int, dims: dict, seed: int, device: torch.device) -> dict:
    """Trainable parameters: one 3×3 filter K_e per cell edge (6, 3, 3); per reduction r a linear read-out
    W_r (10, D_r), b_r (10,) and 10 prototypes P_r (10, D_r) with a log-temperature τ_r ()."""
    gen = torch.Generator().manual_seed(seed)

    def p(t):
        return t.to(device).requires_grad_(True)

    th = {"K": p(0.3 * torch.randn(n_edges, 3, 3, generator=gen))}
    for r, d in dims.items():
        th[f"W_{r}"] = p(0.01 * torch.randn(CLASSES, d, generator=gen))
        th[f"b_{r}"] = p(torch.zeros(CLASSES))
        th[f"P_{r}"] = p(0.5 * torch.randn(CLASSES, d, generator=gen))
        th[f"tau_{r}"] = p(torch.zeros(()))
    return th


def init_cnn(seed: int, device: torch.device, c1: int = 8, c2: int = 16) -> dict:
    """LeNet-style CNN on 14×14: conv 3×3 (1→c1), ReLU, max-pool 2; conv 3×3 (c1→c2), ReLU, max-pool 2 (7 → 3);
    linear (c2·3·3 → 10).
    He-uniform initialisation (as torch.nn's default bound 1/√fan_in for the biases)."""
    gen = torch.Generator().manual_seed(seed)

    def u(shape, fan_in, gain=6.0):
        b = math.sqrt(gain / fan_in)
        return ((torch.rand(shape, generator=gen) * 2 - 1) * b).to(device).requires_grad_(True)

    return {"W1": u((c1, 1, 3, 3), 9), "b1": u((c1,), 9, 1.0), "W2": u((c2, c1, 3, 3), 9 * c1),
            "b2": u((c2,), 9 * c1, 1), "W3": u((CLASSES, c2 * 9), c2 * 9), "b3": u((CLASSES,), c2 * 9, 1)}  # fmt: skip


def cnn_param_count(c1: int = 8, c2: int = 16) -> int:
    """9 c1 + c1 + 9 c1 c2 + c2 + 9 c2 · 10 + 10."""
    return 10 * c1 + 9 * c1 * c2 + c2 + 90 * c2 + 10


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/ai/operator_search_mnist/solve.py
r"""Search, discretise, retrain, write the equation, read it back; train the CNN; run the data study.

Search: first-order DARTS (Liu, Simonyan & Yang 2019) with alternating updates. Each step takes a class-balanced
mini-batch from the train half and updates the operator parameters θ (filters, read-outs, prototypes) with α fixed,
then a mini-batch from the validation half and updates α with θ fixed; the second-order term of DARTS (the gradient
through one unrolled θ step) is dropped. The mixture is softmax(α/T) with the temperature T annealed geometrically
from 1 to 0.001, so the supernet ends as one discrete DAG (this closes the gap between the mixture and the DAG kept,
as the annealed relaxations of SNAS and GDAS do; plain DARTS keeps T = 1). Data constants (class means μ, principal
axes U, feature mean m and scale s) come from each training mini-batch (detached, like batch norm) and from all the
training images for evaluation.

Discretise: every edge keeps its argmax operator; 'zero' may win and removes the edge (DARTS instead excludes zero
and keeps the 2 strongest inputs of every node; with an annealed supernet that rule drops edges the supernet uses).
Retrain: the discrete DAG inherits the parameters the search trained and is fine-tuned briefly on the train half; the
step with the best validation accuracy is kept (early stopping). The CNN is trained with the same early stopping, on
the same images.
"""

import re
import time

import torch
import torch.nn.functional as F

from . import equation as eq
from . import tensors as T

LR_THETA, LR_ALPHA, LR_CNN = 5e-3, 1e-2, 3e-3
EVAL_EVERY = 10
TEMP = (1.0, 0.001)  # softmax temperature of the mixture, annealed geometrically over the search


def sync(dev: torch.device) -> None:
    if dev.type == "cuda":
        torch.cuda.synchronize()


def statistics(x: torch.Tensor, y: torch.Tensor, w: torch.Tensor, th: dict, reds: list[str]) -> dict:
    with torch.no_grad():
        return eq.data_statistics(eq.pooled(x, w, th), y, reds)


def batch_forward(x, y, w, th, reds):
    """Logits of a training batch with the data constants taken from that same batch (detached, like batch norm):
    no stale constants while θ moves. Returns the logits and the constants."""
    Z = eq.pooled(x, w, th)
    st = eq.data_statistics(Z, y, reds)
    return eq.logits(Z, w, th, st), st


def accuracy(pred: torch.Tensor, y: torch.Tensor) -> float:
    return float((pred == y).double().mean())


def predict(x: torch.Tensor, w: torch.Tensor, th: dict, st: dict, chunk: int = 500) -> torch.Tensor:
    with torch.no_grad():
        return torch.cat([eq.forward(x[i : i + chunk], w, th, st).argmax(1) for i in range(0, len(x), chunk)])


# ---------------------------------------------------------------- the search (supernet)
def search(xt, yt, xv, yv, steps: int, batch: int, seed: int, snapshots: int = 16) -> dict:
    """First-order DARTS on the small set. Returns α, θ, the data constants and the mixture weights over training."""
    dev = xt.device
    mask = eq.family_mask().to(dev)
    alpha = torch.zeros(len(eq.EDGES), len(eq.OPS), device=dev, requires_grad=True)
    th = T.init_supernet(len(eq.CELL_EDGES), eq.reduce_dims(), seed, dev)
    opt_t = torch.optim.Adam(th.values(), lr=LR_THETA)
    opt_a = torch.optim.Adam([alpha], lr=LR_ALPHA)
    gen = torch.Generator().manual_seed(seed)
    reds = list(eq.RED_OPS)
    at = sorted({round(k * steps / max(1, snapshots - 1)) for k in range(snapshots)})
    hist = {"step": [], "w": [], "loss": [], "temp": []}
    w = eq.mixture_weights(alpha / TEMP[0], mask)
    eq.init_heads(th, statistics(xt, yt, w, th, reds), reds)
    for step in range(steps + 1):
        temp = TEMP[0] * (TEMP[1] / TEMP[0]) ** (step / max(1, steps))
        w = eq.mixture_weights(alpha / temp, mask)
        if step in at:
            hist["step"].append(step)
            hist["w"].append(w.detach().cpu())
            hist["temp"].append(temp)
        if step == steps:
            break
        i = T.balanced_batch(yt, batch, gen)
        out, st = batch_forward(xt[i], yt[i], w.detach(), th, reds)  # θ step, α fixed
        loss = F.cross_entropy(out, yt[i])
        opt_t.zero_grad()
        loss.backward()
        opt_t.step()
        j = T.balanced_batch(yv, batch, gen)
        wa = eq.mixture_weights(alpha / temp, mask)  # α step, θ fixed; constants from this step's training batch
        loss_a = F.cross_entropy(eq.forward(xv[j], wa, th, st), yv[j])
        (alpha.grad,) = torch.autograd.grad(loss_a, [alpha])  # only ∂L/∂α: no weight gradients of θ computed
        opt_a.step()
        hist["loss"].append(float(loss_a.detach()))
    st = statistics(xt, yt, w.detach(), th, reds)
    return {"alpha": alpha.detach(), "w": w.detach(), "theta": th, "stats": st, "hist": hist}


def discretise(w: torch.Tensor) -> dict:
    """The discrete DAG from the mixture weights w (9, 23): every edge keeps its largest-weight operator. Unlike DARTS,
    'zero' may win (the edge is then removed) and there is no top-2-inputs rule: with the annealed temperature the
    supernet ends close to exactly this DAG. A node whose every input lost to 'zero' keeps its strongest other edge.
    Returns the kept cell edges, pool, reduce and head, and the one-hot w."""
    w = w.detach().cpu()
    nonzero = [eq.OPS.index(o) for o in eq.CELL_OPS[1:]]
    edges = []
    for i in range(1, eq.N_NODES + 1):
        inc = [e for e, (_, ii) in enumerate(eq.CELL_EDGES) if ii == i]
        keep = [(e, eq.OPS[int(w[e].argmax())]) for e in inc if int(w[e].argmax()) != 0]
        if not keep:  # every input of z_i lost to 'zero': keep its strongest non-zero edge (no empty node)
            e = max(inc, key=lambda e: float(w[e, nonzero].max()))
            keep = [(e, eq.OPS[nonzero[int(w[e, nonzero].argmax())]])]
        edges += keep
    edges.sort()
    pick = {name: eq.OPS[int(w[eq.EDGES.index(name)].argmax())] for name in ("pool", "reduce", "head")}
    arch = {"edges": edges, **pick}
    return {**arch, "onehot": onehot(arch)}


def onehot(arch: dict) -> torch.Tensor:
    W = torch.zeros(len(eq.EDGES), len(eq.OPS))
    for e, op in arch["edges"]:
        W[e, eq.OPS.index(op)] = 1.0
    for name in ("pool", "reduce", "head"):
        W[eq.EDGES.index(name), eq.OPS.index(arch[name])] = 1.0
    return W


def n_trainable(arch: dict) -> int:
    """Learned numbers of the discrete DAG: 9 per learned filter, plus the head (10 D + 10 linear, 10 D + 1
    prototypes)."""
    D = eq.reduce_dims()[arch["reduce"]]
    conv = 9 * sum(op == "conv3" for _, op in arch["edges"])
    return conv + (10 * D + 10 if arch["head"] == "linear" else 10 * D + 1)


def n_constants(arch: dict) -> int:
    """Numbers computed in closed form from the training images (no gradient): μ or U when used, m (D,) and one
    scale s per channel."""
    D = eq.reduce_dims()[arch["reduce"]]
    extra = {"template": 10 * eq.CHANNELS * 49, "pca": eq.CHANNELS * 49 * T.Q_PCA}.get(arch["reduce"], 0)
    return D + eq.CHANNELS + extra


def retrain(arch: dict, xt, yt, xv, yv, steps: int, batch: int, seed: int, inherit: dict | None = None) -> dict:
    """Train the discrete DAG's parameters on the train half (Adam), keeping the step with the best validation
    accuracy. With `inherit` (the supernet's θ) it starts from the parameters the search trained (weight inheritance,
    as in one-shot NAS) and is a brief fine-tune; without, from fresh filters and class-mean heads."""
    dev = xt.device
    W = arch["onehot"].to(dev)
    reds = [arch["reduce"]]
    if inherit is not None:
        th = {k: v.detach().clone().requires_grad_(True) for k, v in inherit.items()}
    else:
        th = T.init_supernet(len(eq.CELL_EDGES), eq.reduce_dims(), seed + 7, dev)
        eq.init_heads(th, statistics(xt, yt, W, th, reds), reds)
    used = [k for k in th if k == "K" or k.endswith("_" + arch["reduce"])]
    opt = torch.optim.Adam([th[k] for k in used], lr=LR_THETA)
    gen = torch.Generator().manual_seed(seed + 7)
    best = (-1.0, None, None)
    for step in range(steps + 1):
        if step % EVAL_EVERY == 0 or step == steps:
            st = statistics(xt, yt, W, th, reds)
            acc = accuracy(predict(xv, W, th, st), yv)
            if acc > best[0]:
                best = (acc, {k: v.detach().clone() for k, v in th.items()}, st)
        if step == steps:
            break
        i = T.balanced_batch(yt, batch, gen)
        loss = F.cross_entropy(batch_forward(xt[i], yt[i], W, th, reds)[0], yt[i])
        opt.zero_grad()
        loss.backward()
        opt.step()
    return {"theta": best[1], "stats": best[2], "val_acc": best[0]}


def train_cnn(xt, yt, xv, yv, steps: int, batch: int, seed: int) -> dict:
    """The CNN with Adam on the train half, best validation step kept (early stopping)."""
    p = T.init_cnn(seed, xt.device)
    opt = torch.optim.Adam(p.values(), lr=LR_CNN)
    gen = torch.Generator().manual_seed(seed + 11)
    best = (-1.0, None)
    for step in range(steps + 1):
        if step % EVAL_EVERY == 0 or step == steps:
            with torch.no_grad():
                acc = accuracy(eq.cnn(xv, p).argmax(1), yv)
            if acc > best[0]:
                best = (acc, {k: v.detach().clone() for k, v in p.items()})
        if step == steps:
            break
        i = T.balanced_batch(yt, batch, gen)
        loss = F.cross_entropy(eq.cnn(xt[i], p), yt[i])
        opt.zero_grad()
        loss.backward()
        opt.step()
    return {"params": best[1], "val_acc": best[0]}


# ---------------------------------------------------------------- the equation writer and its reader
def write_equation(arch: dict) -> list[str]:
    """The discrete DAG as LaTeX lines: one per cell node, then the features, their standardisation and the decision."""
    lines = []
    for i in range(1, eq.N_NODES + 1):
        terms = []
        for e, op in arch["edges"]:
            j, ii = eq.CELL_EDGES[e]
            if ii == i:
                terms.append(eq.LATEX[op].format(a="x" if j == 0 else f"z_{{{j}}}", e=e + 1))
        lines.append(f"z_{{{i}}} = " + (" + ".join(terms) if terms else "0"))
    inner = eq.LATEX[arch["pool"]].format(a="z")
    lines.append("f = " + eq.LATEX[arch["reduce"]].format(a=inner))
    lines.append(r"\hat{f} = (f - m) / s")
    if arch["head"] == "linear":
        lines.append(r"\hat{y} = \arg\max_{k}\, (W \hat{f} + b)_{k}")
    else:
        lines.append(r"\hat{y} = \arg\min_{k}\, \|\hat{f} - p_{k}\|^{2}")
    return lines


def _regex(template: str) -> re.Pattern:
    """A formula template → a regex: {a} becomes a group for the argument, {e} for the edge number."""
    text = re.escape(template.format(a="ARGXQ", e="EDGEXQ"))
    return re.compile(text.replace("ARGXQ", "(?P<a>.+)").replace("EDGEXQ", r"(?P<e>\d+)"))


def _split_terms(s: str) -> list[str]:
    """Split at top-level ' + ' (outside brackets)."""
    out, depth, start, i = [], 0, 0, 0
    while i < len(s):
        depth += s[i] in "([{" and 1 or (s[i] in ")]}" and -1 or 0)
        if depth == 0 and s.startswith(" + ", i):
            out.append(s[start:i])
            start = i = i + 3
            continue
        i += 1
    return out + [s[start:]]


def read_equation(lines: list[str]) -> dict:
    """Parse the written LaTeX back into a DAG (only the lines, nothing from the search): every term is matched
    against the operator formulas. Returns the same structure as discretise() (without the one-hot matrix)."""
    fam = {"cell": [o for o in eq.CELL_OPS if o not in ("identity", "zero")], "pool": eq.POOL_OPS, "reduce": eq.RED_OPS}
    pats = {o: _regex(eq.LATEX[o]) for o in eq.LATEX}
    edges = []
    for line in lines[: eq.N_NODES]:
        lhs, rhs = line.split(" = ", 1)
        i = int(re.fullmatch(r"z_\{(\d)\}", lhs).group(1))
        for term in _split_terms(rhs) if rhs != "0" else []:
            for op in fam["cell"] + ["identity"]:
                m = pats[op].fullmatch(term)
                if m and re.fullmatch(r"x|z_\{\d\}", m.group("a")):
                    j = 0 if m.group("a") == "x" else int(m.group("a")[3:-1])
                    e = eq.CELL_EDGES.index((j, i))
                    assert "e" not in m.groupdict() or int(m.group("e")) == e + 1, "filter index ≠ edge"
                    edges.append((e, op))
                    break
            else:
                raise ValueError(f"cannot read the term {term!r}")
    rhs = lines[eq.N_NODES].split(" = ", 1)[1]
    red = next(o for o in fam["reduce"] if pats[o].fullmatch(rhs))
    inner = pats[red].fullmatch(rhs).group("a")
    pool = next(o for o in fam["pool"] if pats[o].fullmatch(inner) and pats[o].fullmatch(inner).group("a") == "z")
    head = "linear" if r"W \hat{f}" in lines[-1] else "proto" if r"p_{k}" in lines[-1] else None
    assert lines[eq.N_NODES + 1] == r"\hat{f} = (f - m) / s" and head is not None
    return {"edges": sorted(edges), "pool": pool, "reduce": red, "head": head}


def evaluate_equation(lines: list[str], x: torch.Tensor, th: dict, st: dict) -> torch.Tensor:
    """ŷ from the written equation alone: read it, then evaluate it line by line with the operator functions and the
    named numbers (K_e, μ, U, m, s, W, b, p_k)."""
    arch = read_equation(lines)
    with torch.no_grad():
        z = {0: x}
        for i in range(1, eq.N_NODES + 1):
            terms = [eq.cell_op(op, z[eq.CELL_EDGES[e][0]], th["K"][e]) for e, op in arch["edges"]
                     if eq.CELL_EDGES[e][1] == i]  # fmt: skip
            z[i] = terms[0] if terms else torch.zeros_like(x)
            for t in terms[1:]:
                z[i] = z[i] + t
        Z = eq.pool_op(arch["pool"], torch.cat([z[0], z[1], z[2], z[3]], 1))
        r = arch["reduce"]
        fh = (eq.reduce_op(r, Z, st) - st[f"m_{r}"]) / st[f"s_{r}"]
        if arch["head"] == "linear":
            return (fh @ th[f"W_{r}"].T + th[f"b_{r}"]).argmax(1)
        return ((fh[:, None, :] - th[f"P_{r}"][None]) ** 2).sum(-1).argmin(1)


# ---------------------------------------------------------------- one run and the study
def run_tensor(data: dict, tr, va, cfg: dict, seed: int) -> dict:
    xt, yt, xv, yv = data["x_pool"][tr], data["y_pool"][tr], data["x_pool"][va], data["y_pool"][va]
    dev = xt.device
    sync(dev)
    t0 = time.perf_counter()
    s = search(xt, yt, xv, yv, cfg["search_steps"], cfg["batch"], seed)
    arch = discretise(s["w"])
    r = retrain(arch, xt, yt, xv, yv, cfg["retrain_steps"], cfg["batch"], seed, inherit=s["theta"])
    sync(dev)
    secs = time.perf_counter() - t0
    W = arch["onehot"].to(dev)
    pred = predict(data["x_test"], W, r["theta"], r["stats"])
    acc_super = accuracy(predict(data["x_test"], s["w"], s["theta"], s["stats"]), data["y_test"])
    return {"arch": arch, "theta": r["theta"], "stats": r["stats"], "hist": s["hist"], "seconds": secs,
            "acc": accuracy(pred, data["y_test"]), "acc_supernet": acc_super, "pred": pred.cpu(),
            "equation": write_equation(arch), "params": n_trainable(arch), "constants": n_constants(arch)}  # fmt: skip


def run_cnn(data: dict, tr, va, cfg: dict, seed: int) -> dict:
    xt, yt, xv, yv = data["x_pool"][tr], data["y_pool"][tr], data["x_pool"][va], data["y_pool"][va]
    sync(xt.device)
    t0 = time.perf_counter()
    c = train_cnn(xt, yt, xv, yv, cfg["cnn_steps"], cfg["batch"], seed)
    sync(xt.device)
    secs = time.perf_counter() - t0
    with torch.no_grad():
        pred = eq.cnn(data["x_test"], c["params"]).argmax(1)
    return {"seconds": secs, "acc": accuracy(pred, data["y_test"]), "params": T.cnn_param_count(), "pred": pred.cpu()}


def baselines(data: dict, tr, va) -> dict:
    """Parameter-free references from all n images per class (train and validation halves together)."""
    idx = torch.unique(torch.cat([tr, va]))
    x, y = data["x_pool"][idx], data["y_pool"][idx]
    return {"ncm": accuracy(eq.nearest_class_mean(x, y, data["x_test"]), data["y_test"]),
            "pca_lda": accuracy(eq.pca_lda(x, y, data["x_test"]), data["y_test"])}  # fmt: skip


def n_to_reach(sizes, acc_mean, target: float) -> float:
    """Images per class at which the mean accuracy first reaches `target`, linear in log n between sizes (nan if
    never reached)."""
    import math

    for k, a in enumerate(acc_mean):
        if a >= target:
            if k == 0:
                return float(sizes[0])
            a0, a1, n0, n1 = acc_mean[k - 1], a, math.log(sizes[k - 1]), math.log(sizes[k])
            return float(math.exp(n0 + (target - a0) / (a1 - a0) * (n1 - n0)))
    return float("nan")


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/ai/operator_search_mnist/plot.py
"""Animate the search and show the study: the mixture weights softmax(α/T) per edge as the temperature falls, the
discrete DAG they pick, its parameters as images, the written equation, accuracy and time against the number of
training images for the CNN and the tensor DAG (every seed a dot), and a few test digits."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FuncFormatter, LogLocator, NullFormatter
from PIL import Image

from problems._common.plotting import ACCENT, export_plot_data
from problems._common.progress import report
from problems._common.runtime import fast_mode

from . import equation as eq

METHODS = {  # colour, marker, line style, label
    "cnn": (ACCENT[1], "o", "-", "CNN (2 conv + linear)"),
    "tensor_dag": (ACCENT[0], "s", "-", "searched tensor DAG"),
    "pca_lda": (ACCENT[2], "^", "--", "PCA + LDA (closed form)"),
    "nearest_class_mean": (ACCENT[3], "D", ":", "nearest class mean"),
}
NODE_XY = {0: (0.07, 0.5), 1: (0.33, 0.86), 2: (0.33, 0.14), 3: (0.6, 0.5)}
EDGE_LABEL_XY = {0: (0.13, 0.76), 1: (0.13, 0.24), 2: (0.40, 0.5), 3: (0.30, 0.56), 4: (0.52, 0.75), 5: (0.52, 0.25)}


def save_gif_shared_palette(fig, update, frames: list[int], path: Path, fps: int = 4) -> Path:
    """Every frame quantised with ONE palette built from a strip of frames, so colours do not drift between frames."""
    fig.set_dpi(80)
    shots = []
    for i, f in enumerate(frames):
        update(f)
        fig.canvas.draw()
        shots.append(Image.fromarray(np.asarray(fig.canvas.buffer_rgba())[..., :3]))
        report(i + 1, len(frames), "frames")
    export_plot_data(fig, path)
    plt.close(fig)
    sample = shots[:: max(1, len(shots) // 12)]
    strip = Image.new("RGB", (sample[0].width, sample[0].height * len(sample)))
    for k, im in enumerate(sample):
        strip.paste(im, (0, k * im.height))
    palette = strip.quantize(colors=255, method=Image.Quantize.MEDIANCUT)
    frames_p = [im.quantize(palette=palette, dither=Image.Dither.NONE) for im in shots]
    tmp = path.with_name(f".{path.stem}.tmp{path.suffix}")
    frames_p[0].save(tmp, save_all=True, append_images=frames_p[1:], duration=int(1000 / fps), loop=0)
    return tmp.replace(path)


def _draw_dag(ax, arch: dict, title: str) -> None:
    ax.clear()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.set_title(title, fontsize=11)
    kept = dict(arch["edges"])
    for e, (j, i) in enumerate(eq.CELL_EDGES):
        (x0, y0), (x1, y1) = NODE_XY[j], NODE_XY[i]
        on = e in kept
        ax.annotate("", (x1, y1), (x0, y0), arrowprops=dict(
            arrowstyle="-|>", lw=1.8 if on else 0.8, color="#1f2937" if on else "#d1d5db", ls="-" if on else "--",
            shrinkA=14, shrinkB=14, connectionstyle="arc3,rad=0.12"))  # fmt: skip
        lx, ly = EDGE_LABEL_XY[e]
        ax.text(lx, ly, eq.SHORT[kept[e]] if on else "0", fontsize=10, ha="center", va="center",
                color="#111827" if on else "#9ca3af",
                bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.9))  # fmt: skip
    for k, (x, y) in NODE_XY.items():
        ax.add_patch(plt.Circle((x, y), 0.055, fc="#dbeafe" if k else "#e5e7eb", ec="#1e3a8a", lw=1.5))
        ax.text(x, y, "$x$" if k == 0 else f"$z_{k}$", ha="center", va="center", fontsize=12)
    steps = [(0.86, "stack $z$\n" + eq.SHORT[arch["pool"]]), (0.5, eq.SHORT[arch["reduce"]]),
             (0.14, eq.SHORT[arch["head"]] + r" $\to\hat y$")]  # fmt: skip
    for y, txt in steps:
        ax.text(0.88, y, txt, ha="center", va="center", fontsize=10,
                bbox=dict(boxstyle="round,pad=0.3", fc="#fef3c7", ec="#b45309"))  # fmt: skip
    ax.annotate("", (0.79, 0.84), (0.66, 0.56), arrowprops=dict(arrowstyle="-|>", color="#b45309", lw=1.4))
    for y0, y1 in ((0.76, 0.58), (0.42, 0.24)):
        ax.annotate("", (0.88, y1), (0.88, y0), arrowprops=dict(arrowstyle="-|>", color="#b45309", lw=1.4))


def _head_image(show: dict) -> tuple[np.ndarray, str, str]:
    """The read-out as an image: for 'vec' the weights (or prototypes) of each digit as 7×7 maps per channel
    (rows x, z1, z2, z3; columns digits 0–9), otherwise the (10, D) matrix."""
    a, th = show["arch"], show["theta"]
    r = a["reduce"]
    M = (th[f"W_{r}"] if a["head"] == "linear" else th[f"P_{r}"]).detach().cpu().numpy()  # (10, D)
    name = "$W_{kj}$" if a["head"] == "linear" else "prototypes $p_k$"
    if r != "vec":
        return M, f"read-out {name} on {eq.SHORT[r]}", "feature $j$"
    C = eq.CHANNELS
    tiles = M.reshape(10, C, 7, 7)
    img = np.full((C * 8 - 1, 10 * 8 - 1), np.nan)
    for k in range(10):
        for c in range(C):
            img[c * 8 : c * 8 + 7, k * 8 : k * 8 + 7] = tiles[k, c]
    return img, f"read-out {name} as 7×7 maps", "digit $k$ (columns)"


def animate(rows: list[dict], sizes, show: dict, data: dict, need: dict, targets, path: Path) -> Path:
    hist = show["hist"]
    W_hist = np.stack([w.numpy() for w in hist["w"]])  # (S, 9, 23)
    mask = eq.family_mask().numpy()
    fig = plt.figure(figsize=(14, 9.2))
    gs = fig.add_gridspec(3, 3, height_ratios=[3.5, 2.8, 1.35], width_ratios=[1.0, 1.15, 1.05], hspace=0.55,
                          wspace=0.42, left=0.085, right=0.955, top=0.95, bottom=0.03)  # fmt: skip

    # α tensor: operators × edges, blank where the typing rule forbids the operator
    ax_a = fig.add_subplot(gs[0, 0])
    cmap = plt.get_cmap("Blues").with_extremes(bad="#f3f4f6")
    shown = lambda w: np.where(mask, w, np.nan).T  # noqa: E731
    im_a = ax_a.imshow(shown(W_hist[0]), cmap=cmap, vmin=0, vmax=1, aspect="auto", interpolation="nearest")
    ax_a.set_yticks(range(len(eq.OPS)), [eq.SHORT[o] for o in eq.OPS], fontsize=9.5)
    ticks = [rf"${j}\to{i}$" for j, i in eq.CELL_EDGES] + ["pool", "reduce", "head"]
    ax_a.set_xticks(range(len(eq.EDGES)), ticks, fontsize=10, rotation=50, ha="right", rotation_mode="anchor")
    ax_a.set_xlabel(r"edge $e$ ($z_j \to z_i$, then the read-out)", fontsize=10)
    ax_a.set_ylabel("operator $o$", fontsize=10)
    ax_a.grid(False)
    cb = fig.colorbar(im_a, ax=ax_a, fraction=0.06, pad=0.02)
    cb.set_label("weight $w_{eo}$ (dimensionless)", fontsize=10)
    cb.ax.tick_params(labelsize=9)
    title_a = ax_a.set_title("", fontsize=11)  # softmax(α/T) at the current step

    ax_d = fig.add_subplot(gs[0, 1])
    from .solve import discretise  # the same rule as the search uses

    # parameters as images
    sub = gs[0, 2].subgridspec(2, 1, height_ratios=[3, 1.2], hspace=0.6)
    ax_h = fig.add_subplot(sub[0])
    img, ttl, xl = _head_image(show)
    v = float(np.nanmax(np.abs(img))) or 1.0
    im_h = ax_h.imshow(img, cmap="RdBu_r", vmin=-v, vmax=v, aspect="auto", interpolation="nearest")
    ax_h.set_title(ttl, fontsize=11)
    ax_h.set_xlabel(xl, fontsize=10)
    if show["arch"]["reduce"] == "vec":
        ax_h.set_xticks(np.arange(10) * 8 + 3, [str(k) for k in range(10)], fontsize=10)
        ax_h.set_yticks(np.arange(eq.CHANNELS) * 8 + 3, ["$x$", "$z_1$", "$z_2$", "$z_3$"], fontsize=10)
        ax_h.set_ylabel("channel", fontsize=10)
    else:
        ax_h.set_yticks(range(10), [str(k) for k in range(10)], fontsize=9)
        ax_h.set_ylabel("digit $k$", fontsize=10)
    ax_h.grid(False)
    cbh = fig.colorbar(im_h, ax=ax_h, fraction=0.05, pad=0.02)
    cbh.set_label("weight (a.u.)", fontsize=10)
    cbh.ax.tick_params(labelsize=9)
    ax_k = fig.add_subplot(sub[1])
    convs = [e for e, op in show["arch"]["edges"] if op == "conv3"]
    if convs:
        K = show["theta"]["K"].detach().cpu().numpy()
        strip = np.full((3, 4 * len(convs) - 1), np.nan)
        for i, e in enumerate(convs):
            strip[:, 4 * i : 4 * i + 3] = K[e]
        vk = float(np.nanmax(np.abs(strip)))
        im_k = ax_k.imshow(strip, cmap="RdBu_r", vmin=-vk, vmax=vk, interpolation="nearest")
        ax_k.set_xticks([4 * i + 1 for i in range(len(convs))], [f"$K_{{{e + 1}}}$" for e in convs], fontsize=10)
        ax_k.set_yticks([])
        ax_k.set_title(f"learned 3×3 filters ({9 * len(convs)} numbers)", fontsize=10)
        cbk = fig.colorbar(im_k, ax=ax_k, fraction=0.05, pad=0.02)
        cbk.set_label("weight (a.u.)", fontsize=9)
        cbk.ax.tick_params(labelsize=8)
        ax_k.grid(False)
    else:
        ax_k.axis("off")
        ax_k.text(0.5, 0.5, "no learned filter chosen:\nevery cell edge is parameter-free", ha="center", va="center",
                  fontsize=10)  # fmt: skip

    # accuracy and time against n
    ax_acc = fig.add_subplot(gs[1, 0])
    ax_t = fig.add_subplot(gs[1, 1])
    rng = np.random.default_rng(0)
    for m, (col, mk, ls, lab) in METHODS.items():
        acc = np.array([[r["acc"] for r in rows if r["n"] == n and r["method"] == m] for n in sizes])
        jit = np.exp(rng.uniform(-0.06, 0.06, acc.shape))
        ax_acc.scatter((np.array(sizes)[:, None] * jit).ravel(), acc.ravel(), s=22, color=col, marker=mk, alpha=0.55,
                       lw=0)  # fmt: skip
        ax_acc.plot(sizes, acc.mean(1), ls, color=col, marker=mk, ms=6, lw=2, label=lab)
        if m in ("cnn", "tensor_dag"):
            t = np.array([[r["seconds"] for r in rows if r["n"] == n and r["method"] == m] for n in sizes])
            ax_t.scatter((np.array(sizes)[:, None] * jit).ravel(), t.ravel(), s=22, color=col, marker=mk, alpha=0.55,
                         lw=0)  # fmt: skip
            ax_t.plot(sizes, t.mean(1), ls, color=col, marker=mk, ms=6, lw=2, label=lab)
    ax_acc.axhline(0.1, color="#9ca3af", lw=1, ls="--")
    ax_acc.text(sizes[-1], 0.12, "chance", fontsize=9, ha="right", color="#6b7280")
    ax_acc.set_xscale("log")
    ax_acc.set_ylim(0, 1)
    ax_acc.set_xlabel("training images per digit $n$ (images)", fontsize=10)
    ax_acc.set_ylabel("test accuracy (dimensionless)", fontsize=10)
    ax_acc.set_title("accuracy on 2000 test digits, every seed a dot", fontsize=11)
    ax_acc.legend(fontsize=9, loc="lower right")
    ax_acc.tick_params(labelsize=10)
    ax_t.set_xscale("log")
    ax_t.set_yscale("log")
    ts = [r["seconds"] for r in rows if r["seconds"] != ""]
    ax_t.set_ylim(min(ts) / 1.5, max(ts) * 1.5)
    ax_t.yaxis.set_major_locator(LogLocator(subs=(1.0, 2.0, 5.0)))
    ax_t.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
    ax_t.yaxis.set_minor_formatter(NullFormatter())
    ax_t.set_xlabel("training images per digit $n$ (images)", fontsize=10)
    ax_t.set_ylabel("training wall time (s)", fontsize=10)
    ax_t.set_title("compute: search + fine-tune vs CNN training", fontsize=11)
    ax_t.legend(fontsize=9, loc="upper left")
    ax_t.tick_params(labelsize=10)
    lines = []
    for t in targets:
        f = lambda v: "never" if np.isnan(v) else f"{v:.0f}"  # noqa: E731
        lines.append(f"{t:.0%}: CNN {f(need['cnn', t])}, DAG {f(need['tensor_dag', t])}, "
                     f"LDA {f(need['pca_lda', t])}")  # fmt: skip
    ax_t.text(0.98, 0.04, "images per digit to reach\n" + "\n".join(lines), transform=ax_t.transAxes, fontsize=9,
              ha="right", va="bottom", bbox=dict(fc="white", ec="#d1d5db"))  # fmt: skip

    # a few test digits with both predictions
    sub = gs[1, 2].subgridspec(3, 4, hspace=0.7, wspace=0.1, height_ratios=[0.25, 1, 1])
    ax_lbl = fig.add_subplot(sub[0, :])
    ax_lbl.axis("off")
    ax_lbl.text(0.5, 0.0, f"test digits (models trained with n = {show['n']})", ha="center", va="bottom", fontsize=11)
    x = data["x_test"].cpu().numpy()[:, 0]
    y = data["y_test"].cpu().numpy()
    idx = np.arange(0, len(y), len(y) // 8)[:8] + 7
    for k, i in enumerate(idx):
        ax = fig.add_subplot(sub[1 + k // 4, k % 4])
        ax.imshow(x[i], cmap="gray_r", vmin=0, vmax=1)
        ax.set_xticks([])
        ax.set_yticks([])
        pd, pc = int(show["pred"][i]), int(show["cnn_pred"][i])
        ax.set_title(f"DAG {pd}{'✓' if pd == y[i] else '✗'}\nCNN {pc}{'✓' if pc == y[i] else '✗'}", fontsize=9.5)

    # the written equation
    ax_e = fig.add_subplot(gs[2, :])
    ax_e.axis("off")
    eqs = show["equation"]
    ax_e.text(0.0, 1.0, f"the found equation (n = {show['n']} per digit, seed 0; test accuracy {show['acc']:.3f}, "
              f"{show['params']} learned numbers):", fontsize=11, va="top", transform=ax_e.transAxes)  # fmt: skip
    for k, line in enumerate(eqs):
        col, row = divmod(k, 3)
        ax_e.text(0.01 + 0.5 * col, 0.66 - 0.3 * row, f"${line}$", fontsize=12, va="top", transform=ax_e.transAxes)

    snaps = list(range(len(W_hist)))
    if fast_mode():
        snaps = snaps[:: max(1, len(snaps) // 3)][:3] + [snaps[-1]]
    hold, n_snap = 2, len(snaps)
    frames = list(range(n_snap * hold + 8))

    def update(f: int):
        k = snaps[min(f // hold, n_snap - 1)]
        im_a.set_data(shown(W_hist[k]))
        title_a.set_text(rf"softmax$(\alpha/T)$: step {hist['step'][k]}, $T$ = {hist['temp'][k]:.3f}")
        arch = discretise(hist["w"][k])
        _draw_dag(ax_d, arch, "argmax DAG now" if k < snaps[-1] else "the discrete DAG kept")

    return save_gif_shared_palette(fig, update, frames, path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'search_steps': 100,
    'retrain_steps': 50,
    'cnn_steps': 200,
    'seeds': 3,
    'batch': 64,
    'show_n': 10
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.ai.operator_search_mnist")]:
    del sys.modules[name]
main = importlib.import_module("problems.ai.operator_search_mnist.main")
problem_dir = Path("problems/ai/operator_search_mnist")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
